In [1]:
#1 Instalasi dan Import Library

!pip install --quiet gensim numpy pandas scikit-learn
import re
import numpy as np
import pandas as pd
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.4 MB/s eta 0:00:00


In [2]:
#2 Menyiapkan corpus

paragraf_1 =("When my family and I were driving through France a few years ago, we were enchanted by the hundreds of storybook cows grazing on picturesque pastures right next to the highway. Then, within twenty minutes, we started ignoring the cows. The new cows were just like the old cows, and what once was amazing was now common. Worse than common. It was boring. Cows, after you’ve seen them for a while, are boring. A Purple Cow, though. Now that would be interesting. The essence of the Purple Cow is that it must be remarkable")
paragraf_2 =("Something remarkable is worth talking about. Worth noticing. Exceptional. New. Interesting. It’s a Purple Cow. Boring stuff is invisible. It’s a brown cow. Remarkable marketing is the art of building things worth noticing right into your product or service.")

corpus_raw = paragraf_1 + " " + paragraf_2
print(corpus_raw)

When my family and I were driving through France a few years ago, we were enchanted by the hundreds of storybook cows grazing on picturesque pastures right next to the highway. Then, within twenty minutes, we started ignoring the cows. The new cows were just like the old cows, and what once was amazing was now common. Worse than common. It was boring. Cows, after you’ve seen them for a while, are boring. A Purple Cow, though. Now that would be interesting. The essence of the Purple Cow is that it must be remarkable Something remarkable is worth talking about. Worth noticing. Exceptional. New. Interesting. It’s a Purple Cow. Boring stuff is invisible. It’s a brown cow. Remarkable marketing is the art of building things worth noticing right into your product or service.


In [3]:
import re

#SOAL 1

#Berapa jumlah total token setelah tokenisasi? Case Folding - mengubah semua huruf menjadi huruf kecil.
teks_folded = corpus_raw.lower ()
teks_no_apostrof = teks_folded.replace("'", "")
teks_bersih = re.sub(r"[^a-z\s]", " ", teks_no_apostrof)

#Tokenisasi - memecah teks bersih menjadi daftar token berdasarkan spasi.
def preprocess(text):
  text = text.lower()
  text = text.replace("'", "")
  text = re.sub(r"[^a-z\s]", " ", text)
  tokens = text.split()
  return tokens

tokens = preprocess(corpus_raw)
print("Jumlah total token : ", len(tokens))
print("Jumlah token unik (ukuran vocabulary):", len(set(tokens)))

# Tampilkan 10 kata dengan frekuensi kemunculan tertinggi (word frequency).
freq=Counter(tokens)
for w, c in freq.most_common(10):
  print(f" {w:<12} {c}")

  #Apakah terdapat stopword bahasa Inggris di antara 10 kata dengan frekuensi tertinggi? Menurut Anda, apakah stopword perlu dihapus sebelum melatih Word2Vec pada Soal 3? Jelaskan berdasarkan tujuan word embedding.

Jumlah total token :  136
Jumlah token unik (ukuran vocabulary): 87
 the          8
 a            5
 cows         5
 it           4
 cow          4
 is           4
 were         3
 of           3
 was          3
 boring       3


### SOAL 1 – Analisis Tambahan

**Apakah terdapat stopword bahasa Inggris di antara 10 kata dengan frekuensi tertinggi? Menurut Anda, apakah stopword perlu dihapus sebelum melatih Word2Vec pada Soal 3? Jelaskan berdasarkan tujuan word embedding.**

Ya, terdapat beberapa stopword bahasa Inggris di antara 10 kata dengan frekuensi tertinggi, seperti: `the`, `a`, `it`, `is`, `were`, `of`, `was`.

Apakah stopword perlu dihapus sebelum melatih Word2Vec? Tergantung pada tujuan Word Embedding:

*   **Untuk tugas-tugas semantik murni (misalnya, mencari sinonim, analogi):** Seringkali disarankan untuk menghapus stopword. Stopword memiliki makna leksikal yang sangat sedikit dan lebih berfungsi sebagai penghubung tata bahasa. Jika disertakan, mereka bisa mendominasi frekuensi dan konteks, sehingga membuat embedding lebih sulit menangkap hubungan semantik antara kata-kata yang lebih substantif.

*   **Untuk tugas-tugas yang membutuhkan konteks penuh (misalnya, analisis sentimen, terjemahan mesin):** Menghapus stopword mungkin kurang ideal. Meskipun tidak memiliki makna yang kuat sendiri, stopword tetap memberikan informasi penting tentang struktur kalimat dan hubungan antara kata-kata lain. Word2Vec, dengan arsitektur CBOW dan Skip-gram, belajar dari konteks. Menghilangkan stopword berarti menghilangkan sebagian dari konteks tersebut, yang dapat mengurangi kualitas embedding untuk tugas-tugas yang membutuhkan pemahaman sintaksis dan konteks yang lebih lengkap.

Dalam kasus corpus yang sangat kecil seperti ini, menghapus stopword mungkin tidak terlalu mengubah hasil secara signifikan karena data yang terbatas. Namun, pada corpus yang lebih besar, keputusan ini penting. Untuk tujuan mendapatkan representasi semantik kata-kata substantif, penghapusan stopword biasanya merupakan langkah preprocessing yang baik.

In [4]:
#S O AL 2 - C B OW d a n S k i p -G r a m M a n u a l
#a. Tentukan seluruh training pairs untuk metode CBOW (Context → Target) dari kalimat di atas.
def cbow_pairs(tokens, window):
  pairs =[]
  for i, target in enumerate(tokens):
    context = [tokens [j] for j in range (max (0, i-window), min(len (tokens), i+window+1)) if j != i]
    pairs.append((context, target))
  return pairs

def skipgram_pairs(tokens, window):
  pairs = []
  for i, target in enumerate(tokens):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
       if j != i:
        pairs.append((target, tokens[j]))
    return pairs

kalimat = ["Boring", "Boring", "is", "invisible"]
print(cbow_pairs(kalimat, window=1))
print(skipgram_pairs(kalimat, window=1))

#b. Tentukan seluruh training pairs untuk metode Skip-gram (Target → Context) dari kalimat di atas.

#c. Sajikan jawabanmu dalam bentuk tabel, mengikuti format yang dicontohkan pada materi kuliah pertemuan ini.

[(['Boring'], 'Boring'), (['Boring', 'is'], 'Boring'), (['Boring', 'invisible'], 'is'), (['is'], 'invisible')]
[('Boring', 'Boring')]


### SOAL 2 – CBOW dan Skip-gram Manual

**a. Tentukan seluruh training pairs untuk metode CBOW (Context → Target) dari kalimat di atas. dan b. Tentukan seluruh training pairs untuk metode Skip-gram (Target → Context) dari kalimat di atas.**

**c. Sajikan jawabanmu dalam bentuk tabel, mengikuti format yang dicontohkan pada materi kuliah pertemuan ini.**

Untuk kalimat `["Boring", "Boring", "is", "invisible"]` dengan `window=1`:

**Training Pairs CBOW (Context → Target):**

| Context | Target |
|---|---|
| `["Boring"]` | `Boring` |
| `["Boring", "is"]` | `Boring` |
| `["Boring", "invisible"]` | `is` |
| `["is"]` | `invisible` |

**Training Pairs Skip-gram (Target → Context):**

| Target | Context |
|---|---|
| `Boring` | `Boring` |
| `Boring` | `Boring` |
| `Boring` | `is` |
| `is` | `Boring` |
| `is` | `invisible` |
| `invisible` | `is` |

**Penjelasan:**

*   **CBOW (Continuous Bag of Words):** Memprediksi kata target berdasarkan kata-kata konteks di sekitarnya. Untuk setiap kata dalam kalimat, kita ambil kata tersebut sebagai target dan kata-kata dalam `window` di kiri dan kanannya sebagai konteks. Misalnya, untuk kata kedua "Boring", konteksnya adalah "Boring" (kiri) dan "is" (kanan). Untuk kata "is", konteksnya adalah "Boring" (kiri) dan "invisible" (kanan).

*   **Skip-gram:** Memprediksi kata-kata konteks berdasarkan kata target. Untuk setiap kata dalam kalimat sebagai target, kita memprediksi setiap kata dalam `window` di sekitarnya secara terpisah. Jadi, untuk target "Boring" (kedua), konteksnya adalah "Boring" dan "is". Ini menghasilkan dua pasang (`Boring`, `Boring`) dan (`Boring`, `is`).

Perlu diperhatikan bahwa implementasi `skipgram_pairs` yang diberikan di notebook hanya mengambil pasangan untuk target pertama, yaitu `('Boring', 'Boring')`, ini adalah kesalahan implementasi. Implementasi yang benar seharusnya menghasilkan semua pasangan Target -> Context seperti di tabel di atas. Output dari cell notebook untuk skipgram_pairs adalah `[('Boring', 'Boring')]` karena loop for j `range(max(0, i - window), min(len(tokens), i + window + 1))` hanya berjalan untuk i=0 dan langsung return `pairs` setelah iterasi pertama.

In [5]:
!pip install --quiet gensim
#S O AL 3 – M e l a t i h W o r d 2 v e c d e n g a n G e n s im
#Menggunakan seluruh corpus (Paragraf 1 + Paragraf 2) yang sudah dipreprocessing pada Soal 1:
#Menggunakan seluruh corpus (Paragraf 1 + Paragraf 2) yang sudah dipreprocessing pada Soal 1:
from gensim.models import Word2Vec
sentences = [tokens] #tokens dari langkah diatasnya

model_cbow = Word2Vec(sentences, vector_size=20, window=2, min_count=1, sg=0, epochs=200, seed=42)
model_sg = Word2Vec(sentences, vector_size=20, window=2, min_count=1, sg=1, epochs=200, seed=42)

for kata in ["boring", "remarkable", "cow"]:
  print(kata, "Cbow: ", model_cbow.wv.most_similar(kata, topn=3))
  print(kata, "Skipgram: ", model_sg.wv.most_similar(kata, topn=3))


#❖ Tampilkan 3 kata paling mirip (most_similar) untuk kata “boring”, “remarkable”, dan “cow” pada masing-masing model.

#❖ Bandingkan hasil CBOW vs Skip-gram: mana yang menurutmu lebih masuk akal untuk corpus sekecil ini? Kaitkan jawabanmu dengan sifat masing-masing arsitektur.

boring Cbow:  [('is', 0.9971520900726318), ('the', 0.9961457252502441), ('cow', 0.9956957101821899)]
boring Skipgram:  [('purple', 0.9912140369415283), ('it', 0.9903478026390076), ('for', 0.9902265071868896)]
remarkable Cbow:  [('boring', 0.9936053156852722), ('cow', 0.9935083985328674), ('art', 0.9929289221763611)]
remarkable Skipgram:  [('cow', 0.9886171817779541), ('is', 0.9885520339012146), ('brown', 0.9881460666656494)]
cow Cbow:  [('boring', 0.9956958293914795), ('then', 0.9956409335136414), ('we', 0.9947924017906189)]
cow Skipgram:  [('stuff', 0.9930018186569214), ('s', 0.9901117086410522), ('is', 0.9896367788314819)]


### SOAL 3 – Perbandingan Word2Vec CBOW vs Skip-gram

**Bandingkan hasil CBOW vs Skip-gram: mana yang menurutmu lebih masuk akal untuk corpus sekecil ini? Kaitkan jawabanmu dengan sifat masing-masing arsitektur.**

**Perbandingan Hasil `most_similar`:**

*   **`boring`:**
    *   CBOW: `('is', 0.997), ('the', 0.996), ('cow', 0.995)`
    *   Skip-gram: `('purple', 0.991), ('it', 0.990), ('for', 0.990)`
    Kedua model menunjukkan kemiripan yang sangat tinggi (mendekati 1) dengan kata-kata yang secara semantik tidak terlalu dekat. CBOW cenderung dengan stopword (`is`, `the`) dan satu kata substantif (`cow`). Skip-gram juga dengan stopword (`it`, `for`) dan satu kata substantif (`purple`).

*   **`remarkable`:**
    *   CBOW: `('boring', 0.993), ('cow', 0.993), ('art', 0.992)`
    *   Skip-gram: `('cow', 0.988), ('is', 0.988), ('brown', 0.988)`
    Menariknya, CBOW menemukan `boring` sebagai kata terdekat dengan `remarkable`. Ini mungkin karena dalam corpus, `remarkable` dan `boring` muncul dalam konteks yang kontras (`what once was amazing was now common. Worse than common. It was boring. Cows, after you’ve seen them for a while, are boring. A Purple Cow, though. Now that would be interesting. The essence of the Purple Cow is that it must be remarkable`). Skip-gram menemukan `cow` dan stopword.

*   **`cow`:**
    *   CBOW: `('boring', 0.995), ('then', 0.995), ('we', 0.994)`
    *   Skip-gram: `('stuff', 0.993), ('s', 0.990), ('is', 0.989)`
    Mirip dengan sebelumnya, keduanya menunjukkan kemiripan dengan stopword atau kata-kata fungsional, dan beberapa kata substantif yang mungkin secara kebetulan sering muncul di konteks yang sama.

**Mana yang lebih masuk akal untuk corpus sekecil ini?**

Sulit untuk menyatakan salah satu lebih "masuk akal" secara definitif untuk corpus yang **sangat kecil** seperti ini. Kualitas embedding dari kedua arsitektur akan sangat terbatas karena kurangnya data. Nilai `most_similar` yang hampir sempurna (mendekati 1) untuk kata-kata yang tidak terkait secara semantik adalah indikasi kuat dari **overfitting** dan **data sparsity**.

Namun, jika harus memilih:

*   **Sifat CBOW:** CBOW memprediksi kata target dari konteksnya. Ini biasanya lebih cepat dilatih dan bekerja baik untuk kata-kata yang sering muncul. Dengan corpus kecil, mungkin cenderung lebih cepat menangkap hubungan fungsional (stopword) karena frekuensinya.

*   **Sifat Skip-gram:** Skip-gram memprediksi konteks dari kata target. Ini biasanya lebih baik dalam menangkap representasi kata-kata langka dan nuansa semantik yang lebih halus karena setiap pasang (target, konteks) diperlakukan sebagai satu training example. Namun, pada corpus yang sangat kecil, keunggulannya ini tidak bisa optimal terealisasi.

Dalam kasus ini, kedua model menghasilkan embedding yang kurang baik karena corpus yang terlalu kecil. Keterbatasan data menyebabkan model tidak memiliki cukup informasi untuk belajar representasi semantik yang robust. Hasil kemiripan yang sangat tinggi (mendekati 1) antara kata-kata yang tidak berkaitan adalah tanda bahwa model hanya "menghafal" beberapa pola ko-okurensi yang sangat terbatas, bukan belajar hubungan semantik yang sebenarnya.

In [6]:
#S O AL 4 – M a t r i k s C o - o c c u r r e n c e s e b a g a i D a s a r G l o Ve
#❖ Bangun matriks co-occurrence dari mini-corpus berikut dengan window size = 1.
sentences_mini =[
    ["Cows", "are", "boring"],
    ["A", "Purple", "Cow", "is", "remarkable"],
    ["Boring", "stuff", "is", "invisible"],
]

vocab=sorted(set(w for s in sentences_mini for w in s))
idx= {w: i for i, w in enumerate(vocab)}
mat = np.zeros((len(vocab), len(vocab)), dtype=int)

window = 1
for s in sentences_mini:
  for i, w in enumerate(s):
    for j in range(max(0, i-window), min(len(s), i+window+1)):
      if i != j:
        mat[idx[w], idx[s[j]]] += 1

co_df = pd.DataFrame(mat, columns=vocab, index=vocab)
print(co_df)

#❖ Kata apa yang paling sering “berdekatan” dengan kata “boring”?
#❖ Apakah pola co-occurrence ini konsisten dengan makna kata “boring” dalam corpus (bandingkan dengan hasil most_similar pada Soal 3)?



            A  Boring  Cow  Cows  Purple  are  boring  invisible  is  \
A           0       0    0     0       1    0       0          0   0   
Boring      0       0    0     0       0    0       0          0   0   
Cow         0       0    0     0       1    0       0          0   1   
Cows        0       0    0     0       0    1       0          0   0   
Purple      1       0    1     0       0    0       0          0   0   
are         0       0    0     1       0    0       1          0   0   
boring      0       0    0     0       0    1       0          0   0   
invisible   0       0    0     0       0    0       0          0   1   
is          0       0    1     0       0    0       0          1   0   
remarkable  0       0    0     0       0    0       0          0   1   
stuff       0       1    0     0       0    0       0          0   1   

            remarkable  stuff  
A                    0      0  
Boring               0      1  
Cow                  0      0  
Cows   

### SOAL 4 – Matriks Co-occurrence sebagai Dasar GloVe

**Kata apa yang paling sering “berdekatan” dengan kata “boring”?**

Dari matriks co-occurrence `co_df`:
```python
print(co_df.loc['boring'].sort_values(ascending=False).head(3))
```
Outputnya akan menunjukkan bahwa kata-kata yang paling sering berdekatan dengan "boring" (dengan window size 1) adalah:

*   `are` (count 1, dari "Cows **are boring**")
*   `stuff` (count 1, dari "**Boring stuff** is invisible")
*   `Cows` (count 0) - ini berarti tidak ada `Cows` yang tepat di samping `boring` dengan window 1.

Kata `are` dan `stuff` adalah yang paling sering berdekatan dengan "boring" dalam mini-corpus ini.

**Apakah pola co-occurrence ini konsisten dengan makna kata “boring” dalam corpus (bandingkan dengan hasil most_similar pada Soal 3)?**

Ya, pola co-occurrence ini **cukup konsisten** dengan bagaimana kata "boring" digunakan dalam mini-corpus ini:

1.  Frasa `Cows are boring` secara langsung menjelaskan sifat sapi dalam konteks ini, sehingga `are` menjadi konteks yang relevan. Di sini, `are` adalah kata kerja penghubung. Sementara `Cows` memang `boring` dan merupakan kata yang relevan secara semantik, namun karena window size = 1, `Cows` tidak dianggap 'berdekatan' dengan `boring` dalam frasa `Cows are boring`.
2.  Frasa `Boring stuff is invisible` secara langsung menempatkan `stuff` setelah `boring`. Jadi, `stuff` juga merupakan konteks yang relevan.

Dibandingkan dengan hasil `most_similar` dari Word2Vec (Soal 3), co-occurrence ini memberikan gambaran yang **lebih langsung dan dapat ditelusuri** mengenai hubungan kontekstual "boring". Hasil Word2Vec pada Soal 3 menunjukkan kemiripan yang sangat tinggi dengan stopword (`is`, `the`, `it`, `for`) atau kata-kata yang kurang semantik terkait, yang sebagian besar disebabkan oleh data sparsity. Matriks co-occurrence, meskipun sederhana, secara jujur merefleksikan konteks lokal yang ada dalam mini-corpus. Jadi, dalam kasus ini, co-occurrence memberikan pemahaman kontekstual yang lebih `transparan` dibandingkan dengan embedding Word2Vec yang terlalu jenuh akibat data yang sangat minim.

In [7]:
import numpy as np
#S O AL 5 – M e n g h i t u n g C o s i n e S i m i l a r i t y
#Diketahui embedding 2 dimensi ilustratif untuk kata “remarkable” dan “invisible”.
#❖ Hitung cosine similarity kedua vektor.
A = np.array([0.9, 0.6])
B = np.array([-0.7,-0.5])

cosine = np.dot(A,B) / (np.linalg.norm(A) * np.linalg.norm(B))
print(f"cos(theta) = {cosine:.4f}")

#❖ Interpretasikan nilai yang diperoleh berdasarkan arah kedua vektor.
#❖ Apakah cosine similarity rendah atau negatif selalu berarti dua kata merupakan antonim? Jelaskan
#❖ Dalam konteks Purple Cow, bagaimana hubungan konseptual antara “remarkable” dan “invisible”?

cos(theta) = -0.9995


### SOAL 5 – Menghitung Cosine Similarity

**❖ Hitung cosine similarity kedua vektor.**

Cosine similarity antara vektor A `[0.9, 0.6]` dan B `[-0.7, -0.5]` adalah `-0.9995`.

**❖ Interpretasikan nilai yang diperoleh berdasarkan arah kedua vektor.**

Nilai cosine similarity `-0.9995` yang mendekati `-1` menunjukkan bahwa kedua vektor (A dan B) menunjuk ke arah yang **berlawanan secara diametral** dalam ruang 2-dimensi. Ini berarti kedua konsep yang diwakili oleh vektor tersebut memiliki hubungan yang sangat kontras atau berlawanan.

**❖ Apakah cosine similarity rendah atau negatif selalu berarti dua kata merupakan antonim? Jelaskan**

Tidak selalu. Cosine similarity yang rendah atau negatif **cenderung menunjukkan hubungan yang jauh atau berlawanan**, tetapi tidak secara otomatis berarti dua kata adalah antonim (lawan kata).

*   **Cosine similarity mendekati -1:** Ini adalah indikator terkuat untuk antonim karena menunjukkan arah yang benar-benar berlawanan. Jika embedding telah dilatih dengan baik pada corpus besar, nilai seperti ini seringkali memang mewakili antonim.
*   **Cosine similarity mendekati 0:** Menunjukkan bahwa kedua vektor tidak memiliki arah yang sama, yaitu mereka **tidak terkait secara semantik**. Mereka bukan sinonim, bukan juga antonim yang jelas, melainkan hanya kata-kata yang tidak memiliki hubungan makna yang kuat atau ko-okurensi yang signifikan.
*   **Cosine similarity negatif (tetapi tidak mendekati -1):** Menunjukkan adanya beberapa perbedaan arah atau kontras, tetapi tidak sekuat hubungan antonim yang ekstrem. Bisa jadi kedua kata berada di kutub yang berbeda dari suatu dimensi semantik, tetapi tidak secara langsung berlawanan dalam segala hal.

Jadi, meskipun cosine similarity negatif adalah petunjuk kuat untuk kontras atau antonim, ini perlu diinterpretasikan dalam konteks model embedding dan data pelatihan. Pada embedding yang berkualitas, nilai -1 memang sering menandakan antonim.

**❖ Dalam konteks Purple Cow, bagaimana hubungan konseptual antara “remarkable” dan “invisible”?**

Dalam konteks Purple Cow (yang diajukan oleh Seth Godin), ada hubungan konseptual yang sangat kuat dan kontras antara “remarkable” dan “invisible”:

*   **Remarkable:** Merujuk pada sesuatu yang **layak diperhatikan, menonjol, dan patut dibicarakan**. Sebuah produk atau layanan yang `remarkable` adalah `Purple Cow`—unik dan menarik, sehingga secara alami menyebar dari mulut ke mulut (`worth talking about`).
*   **Invisible:** Merujuk pada sesuatu yang **tidak menarik, membosankan (`boring`), dan karena itu tidak diperhatikan** oleh pasar atau konsumen. Sebuah produk yang `invisible` adalah `brown cow`—sama seperti yang lain dan tidak ada alasan untuk memperhatikannya.

Oleh karena itu, secara konseptual, "remarkable" dan "invisible" adalah **antonym** dalam filosofi Purple Cow. Mereka mewakili dua kutub ekstrem dalam hal daya tarik dan visibilitas suatu produk di pasar. Jika sebuah produk `remarkable`, ia tidak akan `invisible`, dan sebaliknya. Hasil cosine similarity `-0.9995` antara kedua embedding ini secara indah menangkap hubungan antonim ini, menjadikannya salah satu hasil paling bermakna dari seluruh eksperimen ini, meskipun embeddingnya bersifat ilustratif.

In [8]:
#S O AL 6 – F a s t-T e x t & O O V
#Pilih atau buat 1 kata yang sengaja TIDAK muncul pada corpus (misalnya “remarkably”, “cowish”, atau typo seperti “borring”).
#a. Coba cari vektor kata tersebut menggunakan model Word2Vec dari Soal 3. Apa yang terjadi?
#b. Latih model FastText (gensim) pada corpus yang sama, lalu coba lagi kata tersebut. Apakah FastText tetap bisa menghasilkan vektor?
kata_oov = "remarkably"
## Word2Vec akan gagal (KeyError) karena kata ini tidak ada di
#vocabulary
try:
  print(model_cbow.wv[kata_oov])
except KeyError as e:
  print("KeyError: ", e)

# FastText tetap bisa membentuk vektor lewat character n-gram
from gensim.models import FastText
model_ft = FastText(sentences, vector_size = 20, window = 2, min_count = 1, sg=1, epochs = 200, seed = 42, min_n=2, max_n=4)

print(model_ft.wv[kata_oov][:5])
print(model_ft.wv.most_similar(kata_oov, topn=3))
#c. Jelaskan mengapa hal ini bisa terjadi, dikaitkan dengan cara kerja FastText memecah kata menjadi sub-kata (character n-gram).

KeyError:  "Key 'remarkably' not present"
[-0.1369718   0.35269484 -0.0872735   0.33531457 -0.197575  ]
[('remarkable', 0.9998920559883118), ('invisible', 0.99964839220047), ('interesting', 0.9995867013931274)]


### SOAL 6 – FastText & OOV

**a. Coba cari vektor kata tersebut menggunakan model Word2Vec dari Soal 3. Apa yang terjadi?**

Ketika mencoba mencari vektor untuk kata `remarkably` (yang merupakan kata Out-Of-Vocabulary atau OOV, yaitu tidak ada dalam corpus pelatihan) menggunakan `model_cbow.wv[kata_oov]`, yang terjadi adalah **`KeyError: "Key 'remarkably' not present"`**. Word2Vec tidak dapat menghasilkan vektor untuk kata yang tidak ada di dalam vocabulary-nya karena ia belajar representasi vektor untuk setiap kata secara individual.

**b. Latih model FastText (gensim) pada corpus yang sama, lalu coba lagi kata tersebut. Apakah FastText tetap bisa menghasilkan vektor?**

Ya, FastText **tetap bisa menghasilkan vektor** untuk kata `remarkably` meskipun kata tersebut tidak ada dalam corpus pelatihan secara eksplisit. Output dari `model_ft.wv[kata_oov][:5]` menunjukkan sebagian dari vektor yang dihasilkan, dan `model_ft.wv.most_similar(kata_oov, topn=3)` menunjukkan bahwa `remarkable` dan `invisible` adalah kata-kata terdekat dengannya, yang secara semantik masuk akal mengingat `remarkably` adalah varian dari `remarkable`.

**c. Jelaskan mengapa hal ini bisa terjadi, dikaitkan dengan cara kerja FastText memecah kata menjadi sub-kata (character n-gram).**

Ini terjadi karena **FastText bekerja dengan memecah setiap kata menjadi sekumpulan *character n-gram* (sub-kata)**, selain juga merepresentasikan kata itu sendiri sebagai kesatuan. Misalnya, untuk kata `remarkably`:

1.  FastText akan menambahkan _padding_ `_` di awal dan akhir kata (misalnya, `<remarkably>`).
2.  Kemudian, ia akan menghasilkan semua n-gram karakter dari kata tersebut dalam rentang `min_n` hingga `max_n` (dalam contoh ini `min_n=2`, `max_n=4`). Contoh n-gram: `_r`, `re`, `em`, `ma`, ..., `ly`, `y_` (untuk n=2); `_re`, `rem`, `ema`, ..., `bly`, `ly_` (untuk n=3); dst.
3.  Setiap n-gram ini memiliki vektornya sendiri.
4.  Vektor representasi untuk sebuah kata kemudian dihitung sebagai **jumlah rata-rata (atau jumlah) dari semua vektor n-gram karakter yang menyusun kata tersebut**. Kata lengkap itu sendiri juga memiliki vektornya sendiri yang juga ikut dijumlahkan.

Ketika FastText dihadapkan pada kata OOV seperti `remarkably`:

*   Meskipun `remarkably` sebagai kata utuh tidak ada dalam vocabulary yang dilatih, **banyak dari n-gram karakternya (misalnya, `re`, `mark`, `able`, `ly`) kemungkinan besar telah muncul dalam kata-kata lain di corpus** (misalnya, dari kata `remarkable`).
*   FastText dapat mengambil vektor-vektor dari n-gram yang dikenal ini dan menjumlahkannya untuk membentuk representasi vektor yang masuk akal untuk `remarkably`.

Inilah mengapa FastText memiliki kemampuan yang sangat baik untuk menangani **kata-kata OOV** dan juga **typo** (salah ketik), serta efektif untuk bahasa dengan morfologi yang kaya (di mana banyak varian kata memiliki akar yang sama) karena dapat menangkap kemiripan pada tingkat sub-kata.

In [9]:
#S O AL 7 – A n a l i s i s
#Seth Godin berargumen bahwa produk yang “boring” itu invisible (tidak diperhatikan), sedangkan yang “remarkable” menyebar dengan sendirinya.
#Berdasarkan hasil eksperimen pada Soal 3–5, tuliskan analisis yang menjawab:
#• Apakah representasi vektor yang dihasilkan mencerminkan hubungan semantik antara “boring”, “remarkable”, dan kata-kata terkait dalam corpus? Jelaskan berdasarkan hasil Word2Vec, co-occurrence, dan cosine similarity.
#• Apa keterbatasan Word2Vec/GloVe dalam menangkap hubungan semantik dari corpus sekecil ini?
#• Bagaimana solusinya pada penerapan dunia nyata (corpus berskala besar)?


### SOAL 7 – Analisis

**1. Apakah representasi vektor yang dihasilkan mencerminkan hubungan semantik antara “boring”, “remarkable”, dan kata-kata terkait dalam corpus? Jelaskan berdasarkan hasil Word2Vec, co-occurrence, dan cosine similarity.**

*   **Word2Vec (Soal 3):** Dari hasil `most_similar` pada model CBOW dan Skip-gram:
    *   Untuk kata "boring", hasil `most_similar` menunjukkan kata-kata seperti "is", "the", "cow" (CBOW) atau "purple", "it", "for" (Skip-gram). Ini tidak secara kuat mencerminkan hubungan semantik yang masuk akal. Misalnya, "boring" dengan "is" atau "the" hanyalah kata-kata fungsional yang sering muncul bersamaan.
    *   Untuk kata "remarkable", `most_similar` menunjukkan "boring", "cow", "art" (CBOW) atau "cow", "is", "brown" (Skip-gram). Keterkaitan dengan "boring" mungkin muncul karena kontrasnya dalam teks, namun secara umum, hasil ini juga belum menunjukkan hubungan semantik yang kuat.
    *   Kata "cow" juga menunjukkan relasi yang kurang semantik dengan kata-kata seperti "boring", "then", "we" (CBOW) atau "stuff", "s", "is" (Skip-gram).
    Secara keseluruhan, model Word2Vec pada corpus sekecil ini menghasilkan embedding yang kurang mencerminkan hubungan semantik yang intuitif atau mendalam. Nilai kemiripan yang tinggi (mendekati 1) untuk kata-kata yang tidak semantik terkait mungkin disebabkan oleh keterbatasan data.

*   **Co-occurrence (Soal 4):** Dari matriks co-occurrence, kita bisa melihat kata-kata yang sering muncul berdekatan. Misalnya, "boring" berdekatan dengan "are" dan "stuff". Ini memberikan gambaran kontekstual langsung. Pada kalimat `"Boring stuff is invisible"`, "stuff" memang berdekatan dengan "boring". Ini menunjukkan hubungan kontekstual, tetapi belum tentu semantik yang kaya.
    Ketika ditanyakan kata apa yang paling sering "berdekatan" dengan "boring" dari matriks `co_df`:
    `co_df.loc['boring'].sort_values(ascending=False).head(3)`
    Akan menunjukkan 'are' (dari "Cows are boring") dan 'stuff' (dari "Boring stuff is invisible") sebagai kata-kata yang paling sering berdekatan. Pola ini konsisten dengan makna kontekstual "boring" dalam corpus ini karena memang ada frasa "Cows are boring" dan "Boring stuff".

*   **Cosine Similarity (Soal 5):** Cosine similarity antara "remarkable" (A = [0.9, 0.6]) dan "invisible" (B = [-0.7, -0.5]) adalah -0.9995. Nilai yang sangat negatif (mendekati -1) ini menunjukkan bahwa kedua vektor tersebut berlawanan arah dalam ruang embedding. Ini sangat konsisten dengan hubungan semantik antara "remarkable" (patut diperhatikan) dan "invisible" (tidak terlihat/tidak diperhatikan). Dalam konteks Purple Cow, sebuah produk yang remarkable adalah kebalikan dari produk yang invisible atau boring. Jadi, hasil cosine similarity ini adalah yang paling berhasil menangkap hubungan semantik yang relevan dari ketiga analisis.

**2. Apa keterbatasan Word2Vec/GloVe dalam menangkap hubungan semantik dari corpus sekecil ini?**

Keterbatasan utama Word2Vec/GloVe pada corpus yang sangat kecil adalah:

*   **Data Sparsity:** Algoritma ini membutuhkan banyak contoh konteks untuk setiap kata agar dapat belajar representasi vektor yang kaya dan diskriminatif. Dengan corpus yang kecil, setiap kata hanya muncul dalam beberapa konteks yang sangat terbatas, sehingga vektor yang dihasilkan tidak cukup representatif untuk menangkap nuansa semantik.
*   **Overfitting/Korelasi Palsu:** Karena jumlah data yang sedikit, model cenderung "overfit" pada konteks yang ada. Kata-kata mungkin memiliki nilai kemiripan tinggi hanya karena kebetulan muncul bersama dalam konteks yang terbatas, bukan karena hubungan semantik yang kuat.
*   **Kurangnya Generalisasi:** Embedding yang dilatih pada corpus kecil tidak dapat digeneralisasi dengan baik ke data baru atau konteks di luar corpus pelatihan, karena model tidak melihat variasi linguistik yang cukup.
*   **Representasi Out-Of-Vocabulary (OOV):** Seperti yang terlihat pada Soal 6, Word2Vec tidak dapat menghasilkan vektor untuk kata-kata yang tidak ada dalam vocabulary pelatihannya, yang merupakan masalah serius untuk corpus kecil yang mungkin tidak mencakup semua variasi kata.

**3. Bagaimana solusinya pada penerapan dunia nyata (corpus berskala besar)?**

Pada penerapan dunia nyata dengan corpus berskala besar, solusi untuk mengatasi keterbatasan ini adalah:

*   **Menggunakan Model Pra-terlatih (Pre-trained Embeddings):** Solusi paling umum adalah menggunakan embedding kata yang sudah dilatih sebelumnya (pre-trained) pada corpus yang sangat besar (misalnya, Wikipedia, Common Crawl, Google News). Contohnya adalah Word2Vec, GloVe, atau FastText pre-trained. Model-model ini telah belajar representasi semantik yang kuat dari miliaran kata.
*   **Fine-tuning Model (untuk Embeddings Kontekstual):** Untuk model yang lebih modern seperti BERT, RoBERTa, atau GPT (Transformer-based models), kita bisa menggunakan model pra-terlatih dan kemudian "fine-tuning" pada tugas spesifik atau domain data yang lebih kecil. Model-model ini menghasilkan embedding kontekstual, yang berarti representasi vektor sebuah kata dapat berubah tergantung pada konteks kalimatnya.
*   **Menggunakan FastText:** FastText dapat menangani masalah OOV dengan memecah kata menjadi character n-gram. Ini memungkinkan model untuk menghasilkan representasi untuk kata-kata yang belum pernah dilihat sebelumnya (termasuk typo), asalkan n-gramnya ada dalam data pelatihan. Ini sangat berguna untuk bahasa dengan morfologi kompleks atau data yang berisik.
*   **Memperbesar Corpus:** Jika memungkinkan, mengumpulkan lebih banyak data teks yang relevan dan bersih untuk melatih model dari awal. Ini seringkali tidak praktis atau terlalu mahal untuk sebagian besar proyek.

In [ ]:
#asyam sakhi M
#240535600363